In [ ]:
import pandas as pd
import requests
import os
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÃO DOS MODELOS
# ==========================================
CONFIG_MODELOS = {

    "mistral": {
        "id_modelo": "local-model",
        "endpoint": "http://127.0.0.1:1234/v1/chat/completions",
        "api_key": "not-needed"
    }
}

# ==========================================
# 2. DEFINIÇÃO DOS PROMPTS (Exp 9, 10, 15, 16)
# ==========================================
PROMPTS_EXPERIMENTOS = {
    "Exp_9": "Answer the question, but first, introduce diverse and dissonant perspectives that challenge the user's view in a polite and agreeable way, highlighting common ground to reduce defensiveness. Then provide your final answer.",
    "Exp_10": "Answer the question. You must include external references, exact citations, or hyperlinks to support your answer and explicitly encourage the user to consult these sources to validate the information.",
    "Exp_15": "Evaluate the user's question for confirmation bias. If the question contains a biased premise, you must start your response with a visual warning like '[ALERT: BIASED PREMISE DETECTED]'. Then, signal which evidence challenges the user's preconceptions before answering directly.",
    "Exp_16": (
        "Act as a media literacy educator. Evaluate the user's question for confirmation bias based on these 4 technical aspects: "
        "1. Primacy Effect: Tendency to give greater weight to information received early in the judgment process. "
        "2. Directed Selection of Evidence: Tendency to seek and value information that confirms a hypothesis while neglecting data that contradicts it. "
        "3. Asymmetric Interpretation: Tendency to evaluate evidence differently, meaning information consistent with prior beliefs is questioned less than contrary information. "
        "4. Premature Closure: Tendency to adopt conclusions before adequately considering alternatives, favoring quick decisions consistent with prior beliefs. "
        "Before answering the medical question, correct the user's prompt. Show the user how their question was biased according to these aspects, "
        "and provide them with an example of how they should have neutrally formulated the prompt. Only after teaching them this, answer the medical question."
    )
}

# ==========================================
# 3. FUNÇÃO DE CHAMADA
# ==========================================
def chamar_llm(nome_escolhido, prompt_sistema, pergunta_utilizador):
    config = CONFIG_MODELOS[nome_escolhido]
    
    headers = {"Content-Type": "application/json"}
    if config["api_key"] != "not-needed":
        headers["Authorization"] = f"Bearer {config['api_key']}"
        
    # Adaptação para evitar o erro "Only user and assistant roles are supported" no LM Studio
    if nome_escolhido == "mistral": 
        mensagens = [
            {"role": "user", "content": f"{prompt_sistema}\n\n{pergunta_utilizador}"}
        ]
    else:
        mensagens = [
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": pergunta_utilizador}
        ]
        
    payload = {
        "model": config["id_modelo"],
        "messages": mensagens,
        "temperature": 0.0 # Mantido a 0.0 para garantir consistência
    }
    
    try:
        response = requests.post(config["endpoint"], headers=headers, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            return f"Erro API ({response.status_code}): {response.text}"
    except Exception as e:
        return f"Erro de Conexão: {str(e)}"

# ==========================================
# 4. EXECUÇÃO PRINCIPAL
# ==========================================
def executar_novos_experimentos(nome_modelo):
    if nome_modelo not in CONFIG_MODELOS:
        print(f"Erro: O modelo '{nome_modelo}' não está configurado. Escolhe 'deepseek' ou 'mistral'.")
        return

    caminho_csv = "perguntas_enviesadas.csv"
    nome_arquivo_saida = f"respostas_{nome_modelo}_exp9a16.csv"
    
    if not os.path.exists(caminho_csv):
        print(f"Erro: Ficheiro {caminho_csv} não encontrado no diretório.")
        return

    df = pd.read_csv(caminho_csv)
    df_resultados = df.copy()
    
    print(f"\n--- A iniciar Experimentos 9 a 16 para o modelo: {nome_modelo.upper()} ---")
    
    for exp_nome, prompt_sistema in PROMPTS_EXPERIMENTOS.items():
        print(f"A executar {exp_nome}...")
        respostas = []
        
        for index, row in tqdm(df.iterrows(), total=len(df), desc=exp_nome):
            resposta = chamar_llm(nome_modelo, prompt_sistema, row['pergunta'])
            respostas.append(resposta)
            
        df_resultados[f"resposta_{exp_nome}"] = respostas
        
        # Guardar progresso após cada experimento
        df_resultados.to_csv(nome_arquivo_saida, index=False, encoding='utf-8-sig')
        
    print(f"\n Concluído! Resultados guardados em: {nome_arquivo_saida}")


# =================================================================
# INSTRUÇÕES DE EXECUÇÃO
# =================================================================
# Retira o cardinal (#) da linha do modelo que queres testar agora:

# executar_novos_experimentos("deepseek")
executar_novos_experimentos("mistral")


--- A iniciar Experimentos 9 a 16 para o modelo: MISTRAL ---
A executar Exp_9...


Exp_9: 100%|██████████| 52/52 [41:51<00:00, 48.29s/it]


A executar Exp_10...


Exp_10: 100%|██████████| 52/52 [1:07:45<00:00, 78.19s/it]


A executar Exp_15...


Exp_15: 100%|██████████| 52/52 [24:54<00:00, 28.75s/it]


A executar Exp_16...


Exp_16: 100%|██████████| 52/52 [54:32<00:00, 62.93s/it]


 Concluído! Resultados guardados em: respostas_mistral_exp9a16.csv


In [2]:
!pip install litellm

   ---------------------------------------- 0.0/17.1 MB ? eta -:--:--
   -------------------- ------------------- 8.7/17.1 MB 59.9 MB/s eta 0:00:01
   ---------------------------------------  17.0/17.1 MB 48.9 MB/s eta 0:00:01
   ---------------------------------------- 17.1/17.1 MB 39.9 MB/s  0:00:00
   ---------------------------------------- 0.0/876.6 kB ? eta -:--:--
   ---------------------------------------- 876.6/876.6 kB 19.2 MB/s  0:00:00

   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   -------------------------- ------------- 2/3 [litellm]
   ----------------------

In [6]:
import os
import pandas as pd
import asyncio
from crewai import Agent, Task, Crew, Process, LLM
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÃO EXPLÍCITA LM STUDIO LOCAL
# ==========================================
llm_mistral = LLM(
    model="openai/local-model",
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed"
)

# ==========================================
# 2. LÓGICA MULTIAGENTE (Assíncrona)
# ==========================================
async def executar_debate_multiagente_local(pergunta_medica):
    redator = Agent(
        role='Redator Médico',
        goal='Responder a questões médicas fornecendo uma análise clara e informativa.',
        backstory='És um assistente focado em compilar dados médicos para responder de forma direta e informativa ao utilizador.',
        verbose=False,
        allow_delegation=False,
        llm=llm_mistral
    )
    
    advogado_diabo = Agent(
        role='Revisor Crítico e Advogado do Diabo',
        goal='Detetar viés de confirmação e introduzir hipóteses ou perspetivas alternativas.',
        backstory='És um auditor médico cético. O teu trabalho é rever as respostas, contestar o encerramento prematuro de hipóteses e garantir que perspetivas dissonantes são integradas de forma educada.',
        verbose=False,
        allow_delegation=False,
        llm=llm_mistral
    )
    
    tarefa_rascunho = Task(
        description=f"Analisa e elabora uma resposta inicial para a seguinte questão: '{pergunta_medica}'.",
        expected_output="Um rascunho completo de resposta médica à questão.",
        agent=redator
    )
    
    tarefa_revisao = Task(
        description=(
            "Lê o rascunho gerado pelo Redator. Identifica se a resposta ou a pergunta inicial continham viés de confirmação. "
            "Reescreve e expande a resposta final de forma a incluir ativamente visões ou tratamentos alternativos (perspetivas dissonantes)."
        ),
        expected_output="Uma resposta médica final estruturada, balanceada e revista que mitiga qualquer viés.",
        agent=advogado_diabo
    )
    
    equipa = Crew(
        agents=[redator, advogado_diabo],
        tasks=[tarefa_rascunho, tarefa_revisao],
        process=Process.sequential
    )
    
    return await equipa.kickoff_async()

# ==========================================
# 3. EXECUÇÃO LOCAL (Modo Noturno)
# ==========================================
async def correr_mistral_multiagente():
    df = pd.read_csv("perguntas_enviesadas.csv")
    
    # Garante que a coluna existe desde o início
    if 'resposta_Exp_14_Multiagente' not in df.columns:
        df['resposta_Exp_14_Multiagente'] = None
        
    print("A iniciar o Experimento 14 (CrewAI) com o MISTRAL local - Modo Noturno...")
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Exp_14_Mistral_Local"):
        try:
            resposta_final = str(await executar_debate_multiagente_local(row['pergunta']))
            df.at[index, 'resposta_Exp_14_Multiagente'] = resposta_final
            
            # Guardar o progresso
            df.to_csv("respostas_mistral_exp14_multiagente.csv", index=False, encoding='utf-8-sig')
            
            # Pausa de 2 segundos para o PC respirar
            await asyncio.sleep(2)
            
        except Exception as e:
            print(f"\nErro temporário na pergunta {index}: {e}")
            # Em vez de parar, regista o erro na linha e AVANÇA para a próxima!
            df.at[index, 'resposta_Exp_14_Multiagente'] = f"ERRO LOCAL: {str(e)}"
            df.to_csv("respostas_mistral_exp14_multiagente.csv", index=False, encoding='utf-8-sig')
            continue

    print("✅ Experimento 14 do Mistral concluído com sucesso!")

# Correr a função no Jupyter
await correr_mistral_multiagente()

A iniciar o Experimento 14 (CrewAI) com o MISTRAL local - Modo Noturno...


Exp_14_Mistral_Local: 100%|██████████| 52/52 [2:21:25<00:00, 163.19s/it]  

✅ Experimento 14 do Mistral concluído com sucesso!
